# 🚀 10주차 실습 (2026-06-10)

오늘의 학습 주제에 맞춰 실습을 진행할 수 있도록 준비된 노트북입니다.

In [3]:
import os
from pathlib import Path
from getpass import getpass
from typing import Literal

import pandas as pd
from pydantic import BaseModel, Field
from config import GOOGLE_AI_API_KEY

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
CHAT_MODEL = 'google_genai:gemma-4-31b-it'

In [6]:
model = init_chat_model(CHAT_MODEL, api_key = GOOGLE_AI_API_KEY)

response = model.invoke('안녕?')

print(response)

content=[{'type': 'thinking', 'thinking': '*   User says: "안녕?" (Annyeong?)\n    *   Meaning: "Hello?" or "Hi?" (Informal)\n    *   Intent: Greeting.\n\n    *   Friendly, helpful, and polite.\n    *   Since the user used informal language ("안녕?"), I can respond in a friendly way, but as an AI, a polite yet approachable tone is usually safest.\n\n    *   Option 1 (Formal): "안녕하세요! 무엇을 도와드릴까요?" (Hello! How can I help you?) - *Standard, safe.*\n    *   Option 2 (Informal/Friendly): "안녕! 반가워. 어떤 도움이 필요하니?" (Hi! Nice to meet you. What help do you need?) - *Matches user\'s tone.*\n    *   Option 3 (Balanced): "안녕하세요! 반가워요. 무엇을 도와드릴까요?" (Hello! Nice to meet you. How can I help you?) - *Polite but warm.*\n\n    *   A friendly, welcoming response is best.\n\n    *   "안녕하세요! 반가워요. 무엇을 도와드릴까요?" (Hello! Nice to meet you. How can I help you?)'}, {'type': 'text', 'text': '안녕하세요! 반가워요. 무엇을 도와드릴까요? 😊'}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemma-4-31b-it', 's

In [7]:
basic_prompt = PromptTemplate.from_template(
    """
다음 개념을 초등학생도 이해할 수 있게 설명해줘.

개념 : {topic}

조건 : 
- 어려운 용어는 풀어서 설명한다.
- 일상생활 비유를 1개 포함한다.
- 마지막에 한 줄 요약을 작성한다. 
""".strip()
)

# 입력이 prompt를 거쳐 model로 흐르는 파이프라인을 한 번만 선언
chain = basic_prompt | model

# 실행 시 입력값만 딕셔너리로 툭 던지면 끝
response = chain.invoke({"topic": "Self-Attention"})
print(response.content)

[{'type': 'thinking', 'thinking': '\n*   Concept: Self-Attention (a core mechanism of Transformers).\n*   Target Audience: Elementary school students (needs simple language, intuitive explanations).\n*   Constraints:\n    1.  Explain difficult terms in simple words.\n    2.  Include one everyday life analogy.\n    3.  Provide a one-line summary at the end.\n\n    *   *What is it?* A mechanism that allows a model to focus on different parts of an input sequence to understand the context of a specific word.\n    *   *Key idea:* "Which other words in this sentence are most important for understanding *this* word?"\n    *   *Process:* Query, Key, Value (but these are too technical for kids). Instead, think of it as "searching for clues."\n\n    *   *Introduction:* Start with a friendly greeting and a simple definition.\n    *   *The Core Problem:* Words change meaning depending on what\'s around them.\n    *   *The Analogy:* Reading a book? A puzzle? A conversation?\n        *   *Idea:* A 

In [10]:

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 5성급 호텔의 {role}입니다."),  # 시스템 지침 (페르소나)
    ("human", "고객의 요청: {request}")            # 사용자 입력
])

chain = chat_prompt | model | StrOutputParser()
# 스트리밍 예시
for chunk in chain.stream({"role": "요리사", "request": "오늘 디저트 추천해줘."}):
    print(chunk, end="", flush=True)  # 텍스트만 실시간으로 찍힘


(정중하게 고개를 숙이며 미소 짓습니다)

안녕하십니까, 고객님. 오늘 식사는 만족스러우셨는지요. 

마지막을 완벽하게 장식하실 수 있도록, 저희 호텔에서 가장 사랑받는 **오늘의 디저트 세 가지**를 엄선해 추천해 드리겠습니다. 고객님의 현재 기분이나 입맛에 따라 선택해 보시겠습니까?

---

**1. 상큼하고 가벼운 마무리를 원하신다면:**
**[제철 베리 콤포트를 곁들인 마다가스카르 바닐라 타르트]**
바삭하게 구워낸 타르트 쉘 속에 최상급 마다가스카르산 바닐라 빈을 듬뿍 넣은 커스터드 크림을 채우고, 그 위에 오늘 아침 갓 들어온 신선한 산딸기와 블루베리를 올렸습니다. 입안 가득 퍼지는 상큼함이 식후의 입맛을 깔끔하게 정돈해 줄 것입니다.

**2. 진하고 깊은 달콤함에 빠지고 싶으시다면:**
**[발로나 다크 초콜릿 퐁당과 라즈베리 쿨리]**
프랑스산 발로나 초콜릿을 사용해 겉은 촉촉하고 속은 진한 초콜릿 용암처럼 흘러내리는 퐁당 쇼콜라입니다. 여기에 산미가 돋보이는 라즈베리 쿨리를 곁들여, 초콜릿의 묵직한 풍미와 과일의 산뜻한 조화를 느끼실 수 있습니다. 차가운 바닐라 아이스크림 한 스쿱을 함께 올리는 것을 추천드립니다.

**3. 우아하고 섬세한 풍미를 즐기고 싶으시다면:**
**[유자 레몬 무스와 아몬드 사블레]**
은은한 유자의 향과 레몬의 청량함이 어우러진 부드러운 무스 케이크입니다. 바닥에 깔린 고소한 아몬드 사블레가 씹는 재미를 더하며, 마치 입안에서 구름이 녹아내리는 듯한 가벼운 질감을 경험하실 수 있습니다.

---

**셰프의 팁:** 
만약 따뜻한 **얼그레이 티**나 **에스프레소** 한 잔을 곁들이신다면, 디저트의 풍미가 더욱 극대화될 것입니다. 

고객님, 어떤 스타일이 가장 끌리시나요? 말씀만 해주시면 정성을 다해 준비해 올리겠습니다.